# A full training

# Install

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
# !pip install datasets evaluate transformers[sentencepiece]
# !pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl

# Check GPU is connected

In [ ]:
import torch

In [ ]:
!nvidia-smi

Mon Jul  8 21:09:53 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 552.23                 Driver Version: 552.23         CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA T1000 8GB             WDDM  |   00000000:65:00.0 Off |                  N/A |
| 33%   36C    P8             N/A /   50W |     651MiB /   8192MiB |      7%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

if torch.cuda.is_available():
    print("CUDA is available! 🎉")
    print("Number of CUDA devices:", torch.cuda.device_count())
else:
    print("CUDA is not available. 🙁")

CUDA is available! 🎉
Number of CUDA devices: 1


In [ ]:
import torch
import sys
print('__Python VERSION:', sys.version)
print('__pyTorch VERSION:', torch.__version__)
print('__CUDA VERSION')
from subprocess import call
# call(["nvcc", "--version"]) does not work
! nvcc --version
print('__CUDNN VERSION:', torch.backends.cudnn.version())
print('__Number CUDA Devices:', torch.cuda.device_count())
print('__Devices')
call(["nvidia-smi", "--format=csv", "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free"])
print('Active CUDA Device: GPU', torch.cuda.current_device())
print ('Available devices ', torch.cuda.device_count())
print ('Current cuda device ', torch.cuda.current_device())

__Python VERSION: 3.11.9 | packaged by Anaconda, Inc. | (main, Apr 19 2024, 16:40:41) [MSC v.1916 64 bit (AMD64)]
__pyTorch VERSION: 2.5.0.dev20240626+cu124
__CUDA VERSION
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2021 NVIDIA Corporation
Built on Fri_Dec_17_18:28:54_Pacific_Standard_Time_2021
Cuda compilation tools, release 11.6, V11.6.55
Build cuda_11.6.r11.6/compiler.30794723_0
__CUDNN VERSION: 90100
__Number CUDA Devices: 1
__Devices
Active CUDA Device: GPU 0
Available devices  1
Current cuda device  0


In [ ]:
# activate GPU device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


cuda:0


# Load dataset

In [ ]:
from pathlib import Path
import sys 

# Get the current working directory
cwd = Path.cwd()

# Get the path to the parent directory
parent_dir = cwd.parent

# Add the parent directory to the Python path
sys.path.append(str(parent_dir))

In [ ]:
parent_dir = Path(cwd).parent

# keep in mind that you need to use "/" instead of "\" in the path in windows machine
training_dataset_path = str(parent_dir / 'data/processed/SYNCH/training_set/training_set_0625.csv')
test_dataset_path = str(parent_dir / 'data/processed/SYNCH/test_set/test_set_four_cld.csv')

In [ ]:
from datasets import load_dataset
from datasets import Dataset 
from datasets import DatasetDict
import pandas as pd
import os

In [ ]:
# When training and test set is in one dataset 
my_dataset = pd.read_csv(training_dataset_path)
print(my_dataset.head())
my_dataset = Dataset.from_pandas(my_dataset)

train_test_dataset = my_dataset.train_test_split(test_size=0.5)
test_valid = train_test_dataset['test'].train_test_split(test_size=0.8)

# gather everyone if you want to have a single DatasetDict
raw_datasets = DatasetDict({
    'train': train_test_dataset['train'],
    'test': test_valid['test'],
    'validation': test_valid['train']})


                                           sentence1  \
0                         child being abused in home   
1  community acceptance of intimate partner violence   
2          community intimate partner violence rate    
3             community acceptance of alcohol misuse   
4                      community alcohol misuse rate   

                                           sentence2  label  idx  
0                      adverse childhood experiences      1    0  
1  family and friend permissive intimate partner ...      1    1  
2  family and friend permissive intimate partner ...      1    2  
3                                 sober environments      1    3  
4                                 sober environments      1    4  


In [ ]:

# # Load training set 
# my_dataset = pd.read_csv(training_dataset_path)
# my_dataset = Dataset.from_pandas(my_dataset)

# # Load test set 
# test_set = pd.read_csv(test_dataset_path)
# test_set = test_set.dropna() #TODO: droping NA for now, but there is a problem with the test set
# test_dataset = Dataset.from_pandas(test_set)

# train_test_dataset = my_dataset.train_test_split(test_size=0.1, shuffle=True)
# # test_valid = train_test_dataset['test'].train_test_split(test_size=0.8)

# # raw_train_dataset = raw_datasets["train"]

# # gather everyone if you want to have a single DatasetDict
# raw_datasets = DatasetDict({
#     'train': train_test_dataset['train'],
#     'validation': train_test_dataset['test'],
#     'test': test_dataset})


In [ ]:
# test_set.isna().sum()


In [ ]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 13
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 11
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 2
    })
})

In [ ]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'dire situations',
 'sentence2': 'stressful events',
 'label': 1,
 'idx': 10}

In [ ]:
raw_train_dataset.features

{'sentence1': Value(dtype='string', id=None),
 'sentence2': Value(dtype='string', id=None),
 'label': Value(dtype='int64', id=None),
 'idx': Value(dtype='int64', id=None)}

# Fine tuning a model

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModel
import torch
import torch.nn.functional as F

c:\Users\GL665\AppData\Local\anaconda3\envs\conda_env02\Lib\site-packages\threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


In [ ]:
checkpoint = 'sentence-transformers/all-mpnet-base-v2'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

model.to(device) # Move model to the device

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

# prepare for training

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets['train'].column_names

['labels', 'input_ids', 'attention_mask']

In [ ]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 13
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 11
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2
    })
})

In [ ]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
[len(x) for x in samples["input_ids"]]

[9, 13, 11, 14, 18, 13, 10, 18]

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=8, collate_fn=data_collator
)

In [ ]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 18]),
 'attention_mask': torch.Size([8, 18])}

In [ ]:
# Move the batch of data to the same device
for key in batch.keys():
    batch[key] = batch[key].to(device)

# train the model

## train with trainer

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of MPNetForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-mpnet-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()

  0%|          | 0/6 [00:00<?, ?it/s]

{'train_runtime': 3.2814, 'train_samples_per_second': 11.885, 'train_steps_per_second': 1.828, 'train_loss': 0.670708179473877, 'epoch': 3.0}


TrainOutput(global_step=6, training_loss=0.670708179473877, metrics={'train_runtime': 3.2814, 'train_samples_per_second': 11.885, 'train_steps_per_second': 1.828, 'total_flos': 335055484560.0, 'train_loss': 0.670708179473877, 'epoch': 3.0})

Transform the logits to real predictions

In [ ]:
predictions = trainer.predict(tokenized_datasets["test"])
print(predictions.predictions.shape, predictions.label_ids.shape)

  0%|          | 0/2 [00:00<?, ?it/s]

(11, 2) (11,)


In [ ]:
predictions.predictions

array([[ 4.6038315e-02, -8.2460396e-02],
       [-2.3991674e-02, -5.8233410e-02],
       [ 5.8543168e-02, -6.9986537e-02],
       [-1.6632743e-02, -9.6958978e-03],
       [ 6.9207683e-02, -9.6122593e-02],
       [ 1.5225759e-02, -2.9412506e-03],
       [-6.3943090e-03, -4.8100939e-03],
       [ 5.4414820e-02, -7.7173017e-02],
       [-2.9715590e-02, -7.4894438e-03],
       [ 5.9267059e-02, -4.6326369e-02],
       [ 1.5498906e-02, -4.7661655e-05]], dtype=float32)

using softmax to get the probability of the class. first column is when the class is 0, second column is when the class is 1.. 

In [ ]:
from scipy.special import softmax
probs = softmax(predictions.predictions, axis=1)
print(probs[1:20])

[[0.5085596  0.49144042]
 [0.5320883  0.46791175]
 [0.4982658  0.5017342 ]
 [0.54123867 0.4587613 ]
 [0.50454164 0.49545836]
 [0.49960393 0.5003961 ]
 [0.53284955 0.46715045]
 [0.4944437  0.5055563 ]
 [0.52637386 0.47362614]
 [0.5038866  0.49611345]]


use argmax to get the final predicted class (where the class would be either 0 or 1)

In [ ]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)
print(preds[1:20])

[0 0 1 0 0 1 0 1 0 0]


In [ ]:
validation_dataset = tokenized_datasets["validation"]
validation_dataset[0]

{'labels': tensor(1),
 'input_ids': tensor([    0,  4796,  2004,  2648,  2159,  5406,  2001,  4812,  2002,  6548,
         28620,  8561,     2,     2,  5786,  2000,  4812,  2015,  1041,  4260,
          2034,  2003,  1041,  2159,  2004,  2648,     2]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1])}

## Print validation set

In [ ]:
raw_validation_dataset = raw_datasets["validation"]
raw_validation_dataset[0]



{'sentence1': 'desire to stop family cycle of violence and alcohol misuse',
 'sentence2': 'wanting the violence by a partner or in a family to stop',
 'label': 1,
 'idx': 8}

# evaluate

In [ ]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.2727272727272727, 'f1': 0.2}

# Inference

Let's test the model on a new sentence:

In [ ]:
raw_datasets['validation'][0]

{'sentence1': 'desire to stop family cycle of violence and alcohol misuse',
 'sentence2': 'wanting the violence by a partner or in a family to stop',
 'label': 1,
 'idx': 8}

In [ ]:
sentence1 = "child welfare system intervention"
sentence2 = "children removed from house"
sentence3  = "I am climbing and looking for a climbing partner"

In [ ]:

encoding = tokenizer(sentence1, sentence2, return_tensors="pt")
encoding = {k: v.to(trainer.model.device) for k,v in encoding.items()}

outputs = trainer.model(**encoding)
print(outputs)

SequenceClassifierOutput(loss=None, logits=tensor([[ 0.0544, -0.0772]], device='cuda:0', grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


The logits that come out of the model are of shape (batch_size, num_labels). As we are only forwarding a single sentence through the model, the `batch_size` equals 1. The logits is a tensor that contains the (unnormalized) scores for every individual label.

To turn them into actual predicted labels, we first apply a sigmoid function independently to every score, such that every score is turned into a number between 0 and 1, that can be interpreted as a "probability" for how certain the model is that a given class belongs to the input text.

Next, we use a threshold (typically, 0.5) to turn every probability into either a 1 (which means, we predict the label for the given example) or a 0 (which means, we don't predict the label for the given example).

# Save Pre-trained Model

In [ ]:
dataset_folder_path = 'models/fine_tune_model/02_training_set_0625_def'

cwd = os.getcwd()

parent_dir = Path(cwd).parent


folder_path = str(parent_dir / dataset_folder_path)
print(folder_path)


c:\Users\GL665\Documents\Phrame-MentalMaps\models\fine_tune_model\01_training_set_0625


In [ ]:
trainer.save_model(folder_path)

In [ ]:
# new_model = AutoModelForSequenceClassification.from_pretrained(folder_path)
